# Resume–Job Description Matcher using Transformers 🤖
Semantic matching with `all-MiniLM-L6-v2` and cosine similarity.

## 1. Imports

In [ ]:
import sys
sys.path.append("..")  # allow `import src`

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_resumes, load_jobs
from src.preprocess import clean_text
from src.matcher import load_model, generate_embeddings, similarity_matrix, rank_candidates
from src.visualize import plot_heatmap, plot_ranking

## 2. Data Collection

In [ ]:
resumes = load_resumes("../data/resumes.csv")
jobs = load_jobs("../data/job_descriptions.csv")
print(resumes.shape, jobs.shape)
resumes.head()

In [ ]:
jobs

## 3. Text Preprocessing

In [ ]:
resumes["clean_text"] = resumes["resume_text"].apply(clean_text)
jobs["clean_text"] = jobs["job_description"].apply(clean_text)
resumes[["candidate_name", "clean_text"]].head(3)

## 4. Embedding Generation

In [ ]:
model = load_model()
resume_emb = generate_embeddings(model, resumes["resume_text"].tolist())
job_emb = generate_embeddings(model, jobs["job_description"].tolist())
print("Resume embeddings:", resume_emb.shape)
print("Job embeddings   :", job_emb.shape)

## 5. Cosine Similarity

In [ ]:
sim = similarity_matrix(resume_emb, job_emb)
sim_df = pd.DataFrame(sim, index=resumes["candidate_name"], columns=jobs["job_title"])
sim_df.round(3)

## 6. Candidate Ranking

In [ ]:
job_title = "NLP Engineer"
ranked = rank_candidates(sim_df, job_title)
ranked

## 7. Visualization

In [ ]:
fig = plot_heatmap(sim_df)
fig.savefig("../results/similarity_heatmap.png", dpi=150)
plt.show()

In [ ]:
fig = plot_ranking(ranked, job_title)
fig.savefig("../results/ranking_nlp_engineer.png", dpi=150)
plt.show()

## 8. Best match for every job

In [ ]:
best = sim_df.idxmax(axis=0).rename("best_candidate").to_frame()
best["score"] = sim_df.max(axis=0).round(3)
best

## 9. Save results

In [ ]:
sim_df.round(4).to_csv("../results/similarity_matrix.csv")
ranked.to_csv("../results/ranking_nlp_engineer.csv", index=False)

> ⚠️ Similarity scores represent semantic relevance only, not hiring probability.